# Clasificación de solicitudes de tarjeta

## 1. Objetivo y datos

Se compararán modelos para clasificar la **decisión histórica** asociada a una solicitud: `0` = aprobación y `1` = rechazo. El análisis considerará la detección de rechazos, las falsas alertas y la explicación de las predicciones; la etiqueta no representa impago observado.

El recorrido continúa con calidad y evaluación, exploración y preparación, comparación de modelos, mejoras seleccionadas y evaluación e interpretación. El [README](../README.md) desarrolla el contexto y las reglas de comparación; la [memoria](../docs/Memoria-proyecto.md) recoge las decisiones y resultados.

### Fuentes y carga

Se importan los atributos y las etiquetas desde dos CSV de [Kaggle](https://www.kaggle.com/datasets/rohitudageri/credit-card-details). `Ind_ID` se lee como texto porque representa una clave. Las [instrucciones de descarga](../data/README.md) indican dónde colocar los archivos.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

RAIZ = Path.cwd()
if RAIZ.name == "notebooks":
    RAIZ = RAIZ.parent
DATOS = RAIZ / "data" / "raw"

clientes = pd.read_csv(DATOS / "Credit_card.csv", dtype={"Ind_ID": "string"})
etiquetas = pd.read_csv(DATOS / "Credit_card_label.csv", dtype={"Ind_ID": "string"})

display(pd.DataFrame(
    [clientes.shape, etiquetas.shape],
    index=["Atributos", "Etiquetas"],
    columns=["Registros", "Columnas"],
))

,Registros,Columnas
Atributos,1548,18
Etiquetas,1548,2


### Correspondencia entre tablas

La unión debe asociar una etiqueta a cada registro, sin perder ni multiplicar filas. Se comprueba que las claves sean únicas y estén informadas. La unión externa permite ver también los identificadores presentes solo en una fuente.

In [2]:
for nombre, tabla in {"Atributos": clientes, "Etiquetas": etiquetas}.items():
    if tabla["Ind_ID"].isna().any() or not tabla["Ind_ID"].is_unique:
        raise ValueError(f"{nombre}: revisar identificadores ausentes o repetidos.")

datos = clientes.merge(
    etiquetas, on="Ind_ID", how="outer", validate="one_to_one", indicator=True,
)
display(datos["_merge"].value_counts().rename_axis("Correspondencia").to_frame("Registros"))

if not datos["_merge"].eq("both").all():
    raise ValueError("Hay identificadores sin correspondencia entre las fuentes.")
datos = datos.drop(columns="_merge")

,Registros
Correspondencia,
both,1548
left_only,0
right_only,0


### Primera lectura de los registros

Se muestran cinco registros en columnas, sin identificador, y un resumen de tipos leídos y ausencias. Esta vista permite entender la tabla; todavía no se imputan valores ni se convierten unidades. El [diccionario](../docs/Diccionario-datos.md) aclara los códigos y los significados conocidos.

In [3]:
display(datos.drop(columns="Ind_ID").head().T.rename_axis("Campo"))
display(pd.DataFrame({
    "Tipo leído": datos.dtypes.astype(str),
    "Ausentes": datos.isna().sum(),
}))

,0,1,2,3,4
Campo,,,,,
GENDER,M,F,F,M,F
Car_Owner,Y,Y,N,Y,N
Propert_Owner,Y,Y,Y,Y,Y
CHILDREN,0,2,0,0,2
Annual_income,180000.0,135000.0,247500.0,157500.0,216000.0
Type_Income,Pensioner,Working,Commercial associate,Working,State servant
EDUCATION,Higher education,Secondary / secondary special,Higher education,Secondary / secondary special,Higher education
Marital_status,Married,Married,Separated,Married,Married
Housing_type,House / apartment,House / apartment,Rented apartment,House / apartment,House / apartment


,Tipo leído,Ausentes
Ind_ID,string,0
GENDER,str,7
Car_Owner,str,0
Propert_Owner,str,0
CHILDREN,int64,0
Annual_income,float64,23
Type_Income,str,0
EDUCATION,str,0
Marital_status,str,0
Housing_type,str,0


### Etiqueta y entradas candidatas

Se verifica el dominio de `label` y se muestra la distribución de clases. `X` contendrá los atributos candidatos; `y`, la decisión registrada. La clave se conserva aparte para los controles y las particiones, sin incorporarla al modelo.

In [4]:
if datos["label"].isna().any() or not datos["label"].isin([0, 1]).all():
    raise ValueError("La etiqueta debe estar informada y contener únicamente 0 o 1.")

distribucion = datos["label"].value_counts().sort_index().to_frame("Registros")
distribucion["Porcentaje"] = (100 * distribucion["Registros"] / len(datos)).round(2)
distribucion.index = distribucion.index.map({0: "0 · Aprobación", 1: "1 · Rechazo"})
display(distribucion)

X = datos.drop(columns=["Ind_ID", "label"])
y = datos["label"]
identificadores = datos["Ind_ID"]

,Registros,Porcentaje
label,,
0 · Aprobación,1373,88.7
1 · Rechazo,175,11.3


### Lectura del bloque

Las tablas se emparejan en **1.548 registros**, con **17 atributos candidatos** y una etiqueta. Los **175 rechazos (11,30 %)** muestran que la exactitud por sí sola sería insuficiente para valorar la detección de la clase positiva.

Hay ausencias en género, ingreso, nacimiento y ocupación que requieren interpretación. La clave única no prueba que todos los registros correspondan a personas distintas. La falta de fechas y de una regla documentada de aprobación limita la interpretación a decisiones históricas.

Se conservan los valores de origen. La siguiente sección revisará la calidad y definirá las particiones antes de explorar relaciones o aprender tratamientos.

## 2. Calidad y evaluación

Pendiente de desarrollo: revisar coherencia, códigos y perfiles repetidos, y fijar el procedimiento de validación. Las agrupaciones categóricas y los tratamientos se desarrollarán después sobre entrenamiento.